In [43]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
import logging
import json
from datetime import datetime
from urllib.parse import urljoin
import socket

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('krisha_scraper.log'),
        logging.StreamHandler()
    ]
)

class KrishaScraper:
    def __init__(self, use_proxy=False, test_mode=False):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Referer': 'https://krisha.kz/prodazha/kvartiry/',
            'Connection': 'keep-alive'
        })
        
        self.use_proxy = use_proxy
        self.test_mode = test_mode
        self.data = []
        self.scraped_urls = set()
        self.failed_requests = 0
        
        self.check_internet_connection()
        
    def check_internet_connection(self):
        """Check if we have a working internet connection"""
        try:
            socket.create_connection(("8.8.8.8", 53), timeout=5)
            logging.info("Internet connection is available")
            return True
        except OSError:
            logging.error("No internet connection available")
            return False
            
    def get_proxies(self):
        """Return a list of proxies if needed"""

        return {
            "http": "http://your-proxy-ip:port",
            "https": "http://your-proxy-ip:port"
        }
    
    def make_request(self, url, max_retries=3):
        """Make a request with retry logic"""
        for attempt in range(max_retries):
            try:
                if self.use_proxy:
                    proxies = self.get_proxies()
                    response = self.session.get(url, timeout=30, proxies=proxies)
                else:
                    response = self.session.get(url, timeout=30)
                
                response.raise_for_status()
                return response
                
            except requests.exceptions.RequestException as e:
                logging.warning(f"Request attempt {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    sleep_time = (2 ** attempt) + random.random()
                    logging.info(f"Waiting {sleep_time:.2f} seconds before retry")
                    time.sleep(sleep_time)
                else:
                    logging.error(f"All {max_retries} attempts failed for {url}")
                    raise
        
        return None
    
    def scrape_krisha(self, max_pages=600, delay_range=(2, 5)):
        """Scrape real estate listings from krisha.kz"""
        if self.test_mode:
            return self.test_scraping()
            
        base_url = "https://krisha.kz/prodazha/kvartiry/"
        logging.info(f"Starting Krisha.kz scraping with {max_pages} pages")
        
        for page in range(1, max_pages + 1):
            try:
                # Build URL with pagination
                if page == 1:
                    url = base_url
                else:
                    url = f"{base_url}?page={page}"
                
                logging.info(f"Scraping page {page}: {url}")
                
                response = self.make_request(url)
                if response is None:
                    self.failed_requests += 1
                    if self.failed_requests >= 3:
                        logging.warning("Too many consecutive errors, stopping")
                        break
                    continue
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
     
                if self._is_last_page(soup, page):
                    logging.info("Reached the last page of listings")
                    break
                

                listings = soup.find_all('div', class_='a-card')
                logging.info(f"Found {len(listings)} listings on page {page}")
                
                if not listings:
                    logging.warning(f"No listings found on page {page}")
                    self.failed_requests += 1
                    if self.failed_requests >= 3:
                        logging.warning("Too many consecutive empty pages, stopping")
                        break
                    continue
                

                self.failed_requests = 0
                
                # Process each listing
                for listing in listings:
                    try:
                        property_data = self._parse_listing(listing)
                        if property_data and property_data['url'] not in self.scraped_urls:
                            self.data.append(property_data)
                            self.scraped_urls.add(property_data['url'])
                    except Exception as e:
                        logging.error(f"Error parsing listing: {e}")
                        continue
                
                logging.info(f"Page {page} processed. Total records: {len(self.data)}")
                

                if page % 10 == 0:
                    self.save_data(f"krisha_data_interim_{page}.csv")
                
 
                delay = random.uniform(*delay_range)
                logging.info(f"Waiting {delay:.2f} seconds before next page")
                time.sleep(delay)
                
            except Exception as e:
                logging.error(f"Unexpected error on page {page}: {e}")
                self.failed_requests += 1
                if self.failed_requests >= 3:
                    logging.warning("Too many consecutive errors, stopping")
                    break
                continue
        
        logging.info(f"Krisha scraping completed. Total records: {len(self.data)}")
    
    def test_scraping(self):
        """Test scraping with local HTML files or mock data"""
        logging.info("Running in test mode with mock data")
        

        for i in range(100):
            self.data.append({
                'source': 'krisha.kz',
                'listing_id': f'test_{i}',
                'url': f'https://krisha.kz/test/{i}',
                'title': f'Test Apartment {i}',
                'property_type': 'квартира',
                'rooms': random.randint(1, 1000),
                'size_sq_m': random.randint(0, 1200000),
                'price_kzt': random.randint(0, 500000000000),
                'location': f'Test City, District {i%10}',
                'year_built': random.randint(1900, 2026),
                'seller_type': 'Owner' if i % 2 == 0 else 'Agency',
                'listing_date': '2025-09-19',
                'scraped_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
        
        logging.info(f"Generated {len(self.data)} test records")
    
    def _is_last_page(self, soup, page_num):
        """Check if we've reached the last page of results"""
        # Check for "no results" message
        no_results = soup.find('div', class_='no-search-results')
        if no_results and "ничего не найдено" in no_results.text:
            return True
        

        pagination = soup.find('div', class_='paginator')
        if pagination:
            page_links = pagination.find_all('a')
            if page_links:
                last_page = page_links[-1].get('href')
                if last_page and f'page={page_num}' in last_page:
                    return True
        
        return False
    
    def _parse_listing(self, listing):
        """Parse individual property listing from Krisha"""
        try:
            # Extract URL
            link_elem = listing.find('a', class_='a-card__title')
            if not link_elem or not link_elem.get('href'):
                return None
                
            url = urljoin('https://krisha.kz', link_elem['href'])
            

            title = link_elem.text.strip()
            

            property_type = 'квартира'  
            rooms = None
            size = None
            

            rooms_match = re.search(r'(\d+)-комнатная', title)
            if rooms_match:
                rooms = int(rooms_match.group(1))
            else:

                if 'студия' in title.lower():
                    rooms = 0
            

            size_match = re.search(r'(\d+[,.]?\d*)\s*м²', title)
            if size_match:
                size = float(size_match.group(1).replace(',', '.'))
            
   
            price_elem = listing.find('div', class_='a-card__price')
            price = None
            if price_elem:
                price_text = price_elem.text.strip()
                price = int(re.sub(r'\D', '', price_text)) if price_text else None
            
 
            location_elem = listing.find('div', class_='a-card__subtitle')
            location = location_elem.text.strip() if location_elem else ''
            
            

            params = listing.find_all('div', class_='a-card__text-preview')            
            year = None

            
            for param in params:
                param_text = param.text.strip()
                

                year_match = re.search(r'\b(19|20)\d{2}\b', param_text)
                if year_match:
                    year = int(year_match.group())
            
 
            seller_elem = listing.find('div', class_='a-card__owner')
            seller_type = seller_elem.text.strip() if seller_elem else ''
            

            date_elem = listing.find('div', class_='a-card__stats')
            listing_date = date_elem.text.strip() if date_elem else ''
            

            listing_id = None
            id_match = re.search(r'/(\d+)$', url)
            if id_match:
                listing_id = id_match.group(1)
            
            return {
                'source': 'krisha.kz',
                'listing_id': listing_id,
                'url': url,
                'title': title,
                'property_type': property_type,
                'rooms': rooms,
                'size_sq_m': size,
                'price_kzt': price,
                'location': location,
                'year_built': year,
                'seller_type': seller_type,
                'listing_date': listing_date,
                'scraped_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            
        except Exception as e:
            logging.error(f"Error parsing listing: {e}")
            return None
    
    def remove_duplicates(self):
        """Remove duplicate entries based on URL"""
        if not self.data:
            return
            
        df = pd.DataFrame(self.data)
        initial_count = len(df)
        df.drop_duplicates(subset=['listing_id'], keep='first', inplace=True)
        final_count = len(df)
        self.data = df.to_dict('records')
        logging.info(f"Removed {initial_count - final_count} duplicates. Unique records: {final_count}")
    
    def save_data(self, filename='krisha_data.csv'):
        """Save data to CSV file"""
        if not self.data:
            logging.warning("No data to save")
            return
            
        df = pd.DataFrame(self.data)
        

        columns = [
            'source', 'listing_id', 'url', 'title', 'property_type', 'rooms', 
            'size_sq_m', 'price_kzt', 'location',
            'year_built','seller_type', 
            'listing_date', 'scraped_date'
        ]
        

        columns = [col for col in columns if col in df.columns]
        
        df = df[columns]
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        logging.info(f"Data saved to {filename} with {len(df)} records")
        
    def run_scraping(self, max_pages=5):
        """Run the complete scraping process"""
        try:
            self.scrape_krisha(max_pages=max_pages)
            self.remove_duplicates()
            self.save_data()
            

            if self.data:
                with open('krisha_data.json', 'w', encoding='utf-8') as f:
                    json.dump(self.data, f, ensure_ascii=False, indent=2)
                logging.info("Data also saved as JSON")
                
        except Exception as e:
            logging.error(f"Error during scraping process: {e}")
        finally:

            if self.data:
                self.save_data('krisha_data_final.csv')


if __name__ == "__main__":

    try:
        response = requests.get('https://krisha.kz', timeout=10)
        if response.status_code == 200:
            logging.info("Successfully connected to krisha.kz")

            scraper = KrishaScraper(use_proxy=False, test_mode=False)
        else:
            logging.warning(f"Could not access krisha.kz (status: {response.status_code})")

            scraper = KrishaScraper(use_proxy=False, test_mode=True)
    except requests.exceptions.RequestException as e:
        logging.error(f"Network error when testing connection to krisha.kz: {e}")

        scraper = KrishaScraper(use_proxy=False, test_mode=True)
    
    scraper.run_scraping(max_pages=600)


2025-09-21 19:15:24,695 - INFO - Successfully connected to krisha.kz
2025-09-21 19:15:24,846 - INFO - Internet connection is available
2025-09-21 19:15:24,847 - INFO - Starting Krisha.kz scraping with 600 pages
2025-09-21 19:15:24,847 - INFO - Scraping page 1: https://krisha.kz/prodazha/kvartiry/
2025-09-21 19:15:25,593 - INFO - Found 23 listings on page 1
2025-09-21 19:15:25,593 - INFO - Page 1 processed. Total records: 23
2025-09-21 19:15:25,593 - INFO - Waiting 4.90 seconds before next page
2025-09-21 19:15:30,504 - INFO - Scraping page 2: https://krisha.kz/prodazha/kvartiry/?page=2
2025-09-21 19:15:31,182 - INFO - Found 20 listings on page 2
2025-09-21 19:15:31,197 - INFO - Page 2 processed. Total records: 42
2025-09-21 19:15:31,197 - INFO - Waiting 4.27 seconds before next page
2025-09-21 19:15:35,471 - INFO - Scraping page 3: https://krisha.kz/prodazha/kvartiry/?page=3
2025-09-21 19:15:35,958 - INFO - Found 20 listings on page 3
2025-09-21 19:15:35,973 - INFO - Page 3 processed. 